# 五山地铁站周边餐饮分析

**高德POI采集 + DeepSeek智能分类 + 交互式可视化**

---

## 项目流程

1. 高德API网格化采集 → 原始数据
2. DeepSeek API 批量分类 → 带类别标签数据
3. 生成交互式地图 + 统计图表
4. 分析结论

> 运行前请确保在 `.env` 文件中填写了 `AMAP_KEY` 和 `DEEPSEEK_API_KEY`。

## 环境设置

导入所有依赖模块，检查API配置。

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from config import (
    AMAP_KEY, DEEPSEEK_API_KEY,
    RAW_DATA_FILE, FINAL_DATA_FILE
)
from poi_fetcher import collect_all_shops, save_raw_data
from classifier import classify_all
from visualizer import create_map, create_charts, print_summary

# 检查 API 密钥
def check_keys():
    ok = True
    if not AMAP_KEY or AMAP_KEY == 'your_amap_key_here':
        print('[警告] 高德地图 AMAP_KEY 未配置')
        ok = False
    if not DEEPSEEK_API_KEY or DEEPSEEK_API_KEY == 'your_deepseek_key_here':
        print('[警告] DeepSeek DEEPSEEK_API_KEY 未配置')
        ok = False
    if ok:
        print('API 密钥配置完成')
    return ok

check_keys()

---
## 步骤 1：高德API网格化采集

使用高德地图 `v3/place/around` 接口，以五山地铁站为中心，通过 5 个 500m 半径网格覆盖 1000m 范围，获取所有餐饮类POI数据。

In [ ]:
# 采集所有餐饮POI
df_raw = collect_all_shops()

# 查看前5条数据
df_raw.head()

In [ ]:
# 查看数据概览
print(f"数据形状: {df_raw.shape}")
print(f"列名: {list(df_raw.columns)}")
df_raw.describe()

In [ ]:
# 保存原始数据
save_raw_data(df_raw, RAW_DATA_FILE)

---
## 步骤 2：DeepSeek API 批量分类

调用 DeepSeek 大语言模型，根据店铺名称和地址自动分类到以下 8 个类别：

火锅 | 烧烤 | 茶饮 | 快餐简餐 | 粉面馆 | 地方菜系 | 面包甜点 | 其他

采用批量调用（每10条合并一次请求）+ 结果缓存，减少 API 费用。

In [ ]:
# 执行批量分类
df = classify_all(df_raw)

# 查看分类结果
df[['name', 'address', 'category']].head(10)

In [ ]:
# 保存最终数据
df.to_excel(FINAL_DATA_FILE, index=False, engine='openpyxl')
print(f'最终数据已保存到: {FINAL_DATA_FILE}')

---
## 步骤 3：交互式地图

按类别颜色标记店铺位置（folium Marker），叠加热力图（HeatMap）展示整体聚集程度。

底图使用高德地图瓦片，解决 OpenStreetMap 国内加载问题。

In [ ]:
# 生成交互式地图（在 Notebook 中可直接显示）
m = create_map(df)
m  # 显示地图

---
## 步骤 4：统计图表

生成类别数量柱状图和饼图。

In [ ]:
create_charts(df)

In [ ]:
# 在 Notebook 中显示生成的图表
from IPython.display import Image, display
print("柱状图：")
display(Image(filename='category_bar.png'))
print("饼图：")
display(Image(filename='category_pie.png'))

---
## 分析结论

In [ ]:
print_summary(df)

---

## 下一步可扩展方向

- 接入大众点评评分与人均消费
- 增加时间段分析（早餐/夜宵店铺分布）
- 制作 Dashboard（Streamlit 或 PyEcharts）

---

**Author**：张阳  
**Date**：2026.06